# Mindestanforderungen Klassifikation
- Führen Sie mit dem Algorithmus Ihrer Wahl eine Klassifikationsaufgabe auf Ihren Daten durch.
    - Ziel war die Vorhersage der Zielvariable `Employment`
    - Klassen waren _employed_, _independent contractor / freelancer / self-employed_, _student_ und _not employed_
    - unerwünschte Klassen (`i prefer not to say`, `Other`, `nan`) wurden entfernt
    - Wir haben die **lineare Support Vector Machine** genutzt
___
- Teilen Sie dazu zunächst die Daten auf, um Overfitting beim Trainieren des Algorithmus und bei der Parameterauswahl zu vermeiden. Erklären Sie die gewählte Strategie und die Größenverhältnisse.
    - Wir haben auf 80/20 gesplittet, also 80% Train und 20% Test (`train_test_split(test_size=0.2)`)
    - wir haben `stratify=y` angewandt um eine ähnliche Klassenverteilung in den Train- und Testdaten zu erhalten
    - Hyperparameter-Tuning erfolgt nur auf dem Trainingssplit mittels 3-facher Cross-Validation (`GridSearchCV(cv=3)`)
    - Testsplit bleibt unangetastet und dient finalen, fairen Bewertung
---
- Wählen Sie geeignete Features aus und setzen Sie die Parameter des Algorithmus. Beschreiben Sie das gewälhte Vorgehen für die Auswahl der Features und Parameter. Berichten Sie den Parameterraum und die final gewählten Parameter. Geben Sie die Performanz auf den Trainingsdaten (bzw. Entwicklungsdaten, falls verwendet) an.
    - **Features**:
        - **Datengrundlage:** One-Hot-Encoded Datensatz in dem Features bereits numerisch sind
        - alle `Employment_*`-Spalten werden aus den Features entfernt, da sie sonst die Zielvariable direkt verraten würde
        - Entfernt wurden: `ResponseId`, `AgeNum`, `RemoteCategoryNum` und `ConvertedCompTotal` (Gehälter)
    - **Preprocessing**:
        - fehlende Werte mittels Median-Imputation ersetzt
        - StandardScaler wird eingesetzt
    - **Modell & Parameterwahl**
        - Modell: **LinearSVC** (lineare Support Vector Machine)
        - Hyperparameter-Tuning über `GridSearchCV` mit Optimierungsmaß macro-F1 (jede Klasse zählt gleich, trotz Klassenunwucht)
        - `GridSearchCV` (cv=3) auf Trainingsdaten
        - Optimierungsmaß: macro-F1
    - **Parameterraum**:
        - `C`: 0.5, 1.0, 2.0
        - `class_weight`: None, "balanced"
    - **Beste Parameter**:
        - `C` = 2.0
        - `class_weight` = None
    - **Train-Leistung**
        - CV macro-F1 ~ 0.6824
---
- Evaluieren Sie die Klassifikation auf den ungesehenen Testdaten. Betrachten Sie Precision und Recall sowie den F-Wert. Welches Maß ist für Ihre Anwendung wichtiger? Bewerten Sie Ihr Ergebnis. Ist es in der Praxis voraussichtlich zufriedenstellend?
    - Test-Ergebnisse
        - Accuracy: 0.93
        - macro-F1: 0.77
        - weighted-F1: 0.92
    - Klassenweise Precision / Recall / F1
      employed*: 0.94 / 0.99 / 0.96
        - self-employed (independent contractor / freelancer): 0.92 / 0.58 / 0.71
        - not employed: 0.82 / 0.74 / 0.78
        - student: 0.75 / 0.53 / 0.62
    - Precision/Recall
        - sehr hoch für _employed_
        - gut für _self-employed (independent contractor / freelancer)_
        - geringer für _student_ und _not employed_ (kleine Klassen)
    - wichtigstes Maß
        - macro-F1 ist wichtiger als die Accuracy, da die Klassen sehr stark unbalanciert sind
    - Bewertung
        - das Modell erkennt Mehrheitsklassen sehr zuverlässig
        - Minderheitsklassen werden schwieriger erkannt, hier wären in der Praxis voraussichtlich weitere Maßnahmen nötig

# Codeerklärungen
- imports laden
- csv einlesen
- Zielvariable `Employment` festlegen, die durch Klassifikation vorhergesagt werden soll

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

In [2]:
df = pd.read_csv("One-Hot-Encoded-final.csv")

employment_cols = [c for c in df.columns if c.startswith("Employment_")]
if not employment_cols:
    raise ValueError("Keine Employment_* Spalten gefunden")

emp_mat = df[employment_cols].fillna(0).astype(float)

y = emp_mat.idxmax(axis=1).str.replace("Employment_", "", regex=False)

# Optional: Self-employed zusammenziehen (wie in deiner Beschreibung)
# y = y.replace({"independent contractor, freelancer, or self-employed": "self-employed"})

# Optional: "i prefer not to say" und "nan" entfernen + bad rows entfernen
drop_classes = {"i prefer not to say", "nan", "Other"}
keep_mask = (~y.isin(drop_classes))

df = df.loc[keep_mask].copy()
y = y.loc[keep_mask].copy()

print("Klassenverteilung:\n", y.value_counts())
# print(*df.columns, sep="\n")

Klassenverteilung:
 employed                                                15779
independent contractor, freelancer, or self-employed     2118
student                                                   487
not employed                                              210
Name: count, dtype: int64


- Zeilen ohne `Employment` werden entfernt
- Unerwünschte Klassen werden entfernt, da sie irrelevant sind und zu wenige Beispiele haben (`prefer not to say` & `Other`)
- zudem wird `ResponseId` entfernt, da es nur eine ID ist - kein inhaltlicher Mehrwert
- `AgeNum` ist redundant aufgrund von `Age` (String)
- `ConvertedCompTotal` zu viele leere Zellen/NaN-Werte

In [3]:
drop_feature_cols = employment_cols + ["ResponseId", "AgeNum", "ConvertedCompTotal", "RemoteCategoryNum"]
X = df.drop(columns=drop_feature_cols, errors="ignore")

num_cols = X.select_dtypes(include=["number", "bool", "uint8", "int64", "float64", "int32", "float32"]).columns.tolist()
X = X[num_cols].copy()
print("X shape:", X.shape, "| y shape:", y.shape)
print("Numeric cols:", len(num_cols))


X shape: (18594, 377) | y shape: (18594,)
Numeric cols: 377


- wir splitten in 80% Training und 20% Test
- `stratify=y` -> Klassenverteilung bleibt bei Train und Test ungefähr gleich
- stratify wichtig, weil Zielvariable stark unbalanciert ist

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Preprocessing
- `SimpleImputer(median)` füllt fehlende numerische Werte, also `NaN-Werte`, mit dem Median der Spalte
- StandardScaler skaliert die Zahlen auf vergleichbare Größenordnungen, was Support Vector Machines hilft, da sie empfindlich auf unterschiedliche Skalen reagieren können

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler(with_mean=False)),
        ]), num_cols)
    ],
    remainder="drop"
)

- in der Pipeline kommt nun die Vorverarbeitung und der Klassifikator (LinearSVC) zusammen
- Warum LinearSVC?
    - für Textdaten mit vielen Features funktioniert lineare SVC oft sehr gut

In [6]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LinearSVC(max_iter=20000))
])

- hier werden mehrere sinnvolle Einstellungen getestet:
    - `C`:
        - Regularisierung der SVM: kleiner C entspricht stärkerer Regularisierung, ein größeres C bedeutet mehr Flexibilität
    - `class_weight="balanced"`:
        - wichtig bei unbalancierten Klassen, da so kleine Klassen stärker gewichtet werden

In [7]:
param_grid = {
    "classifier__C": [0.5, 1.0, 2.0],
    "classifier__class_weight": [None, "balanced"],
}

- GridSearchCV testet alle Parameterkombinationen
- Bewertung: `f1_macro`
    - sinnvoll, weil jede Klasse gleich gewichtet wird
- cv=3 bedeutet 3-fache Cross-Validation auf dem Trainingsset
- Testset bleibt unangetastet -> fairer Vergleich

In [8]:
grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

print("\nBest CV macro-F1:", grid.best_score_)
print("Best params:", grid.best_params_)

Fitting 3 folds for each of 6 candidates, totalling 18 fits

Best CV macro-F1: 0.6823739015058177
Best params: {'classifier__C': 2.0, 'classifier__class_weight': None}


- bestes Modell aus der GridSearch wird verwendet
- dann einmalige Evaluierung auf den Testdaten
- Confusion Matrix zeigt, welche Klassen miteinander verwechselt wurden
- erwartungsgemäß wird die Mehrheitsklasse (employed) sehr zuverlässig erkannt
- Minderheitsklassen werden häufiger fälschlich als `employed` klassifiziert

In [9]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("\nTest report:\n", classification_report(y_test, y_pred, zero_division=0))

labels_sorted = best_model.named_steps["classifier"].classes_
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred, labels=labels_sorted))


Test report:
                                                       precision    recall  f1-score   support

                                            employed       0.94      0.99      0.96      3156
independent contractor, freelancer, or self-employed       0.92      0.58      0.71       424
                                        not employed       0.82      0.74      0.78        42
                                             student       0.75      0.53      0.62        97

                                            accuracy                           0.93      3719
                                           macro avg       0.85      0.71      0.77      3719
                                        weighted avg       0.93      0.93      0.92      3719

Confusion matrix:
 [[3128   19    1    8]
 [ 171  248    0    5]
 [   5    2   31    4]
 [  38    2    6   51]]
